# Eval Results Explorer

Load all `results.jsonl` files from `evals/reports/`, visualise quality trends and failure patterns.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sns.set_theme(style='whitegrid')

REPORTS_DIR = Path('../../evals/reports')

In [ ]:
# Load all results.jsonl across all runs
rows = []
for run_dir in sorted(REPORTS_DIR.iterdir()):
    jsonl = run_dir / 'results.jsonl'
    manifest = run_dir / 'manifest.json'
    if not jsonl.exists():
        continue
    run_id = run_dir.name
    commit = ''
    if manifest.exists():
        m = json.loads(manifest.read_text())
        commit = m.get('git_commit', '')
    with jsonl.open() as fh:
        for line in fh:
            r = json.loads(line)
            r['run_id'] = run_id
            r['commit'] = commit
            rows.append(r)

df = pd.DataFrame(rows)
print(f'Loaded {len(df)} case results across {df["run_id"].nunique()} runs')
df.head()

In [ ]:
# Derive difficulty from case_id prefix
def _difficulty(case_id: str) -> str:
    n = int(case_id.split('-')[-1]) if case_id.split('-')[-1].isdigit() else 0
    if case_id.startswith('NEG'):
        return 'negative'
    if n <= 10:
        return 'easy'
    if n <= 20:
        return 'medium'
    return 'hard'

df['difficulty'] = df['case_id'].apply(_difficulty)

In [ ]:
# Heatmap: difficulty × metric
metrics = ['citation_recall', 'citation_precision', 'hallucination_rate',
           'concept_coverage', 'forbidden_claim_rate', 'caveat_coverage', 'legal_quality_score']

pivot = df.groupby('difficulty')[metrics].mean()
pivot = pivot.reindex(['easy', 'medium', 'hard', 'negative'])

fig, ax = plt.subplots(figsize=(12, 4))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='RdYlGn', vmin=0, vmax=1, ax=ax)
ax.set_title('Mean metrics by difficulty')
plt.tight_layout()
plt.show()

In [ ]:
# Temporal evolution of legal_quality_score per run
run_summary = df.groupby('run_id')['legal_quality_score'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(run_summary['run_id'], run_summary['legal_quality_score'], marker='o')
ax.set_xlabel('Run')
ax.set_ylabel('legal_quality_score')
ax.set_title('Legal quality score over time')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
# Failed cases: forbidden_claim_rate > 0 or citation_recall < 0.5
latest_run = df['run_id'].max()
latest = df[df['run_id'] == latest_run]

failed = latest[
    (latest['forbidden_claim_rate'] > 0) |
    (latest['citation_recall'] < 0.5) |
    (latest['passed'] == False)
]

cols = ['case_id', 'difficulty', 'routing_correct', 'citation_recall',
        'hallucination_rate', 'forbidden_claim_rate', 'legal_quality_score', 'error']
print(f'Failed cases in latest run ({latest_run}): {len(failed)}')
failed[cols]